# [Introduction | Online Z3 Guide](https://microsoft.github.io/z3guide/programming/Z3%20Python/advanced#unsat-cores-and-soft-constraints)

In [1]:
from z3 import ArithRef, BoolRef, Reals, solve

## Expressions, Sorts and Declarations

$$
\begin{aligned}
x+1 &: \mathrm{Int}\\
y+1 &: \mathrm{Real}\\
x\ge 2 &: \mathrm{Bool}
\end{aligned}
$$

In [3]:
from z3 import ArithRef, Int, Real

x: ArithRef = Int("x")
y: ArithRef = Real("y")

print((x + 1).sort())  # Int
print((y + 1).sort())  # Real
print((x >= 2).sort())  # Bool

Int
Real
Bool


eq()：AST の構造的同一性

In [4]:
from z3 import ArithRef, Int, Real, eq, simplify

x: ArithRef = Int("x")
y: ArithRef = Int("y")

n: ArithRef = x + y

print(eq(x + y, x + y))  # True
print(eq(x + y, y + x))  # False
print(eq(n, x + y))  # True

x2: ArithRef = Int("x")
print(eq(x, x2))  # True

print(eq(Int("x"), Real("x")))  # False

# 数学的な等しさとの比較
print(simplify((x + y) == (y + x)))  # True

True
False
True
True
False
True


hash()：AST のハッシュ値

In [5]:
from z3 import ArithRef, Int

x: ArithRef = Int("x")

print((x + 1).hash())
print((1 + x).hash())
print(x.sort().hash())

38638296
4274096646
3306638384


$$
\boxed{
\underbrace{x+1}_{\text{Application}}
=
\underbrace{+}_{\text{Declaration}}
\left(
\underbrace{x,1}_{\text{Arguments}}
\right)
}
$$

 three basic groups: **applications**, **quantifiers** and **bounded/free variables**. 

In [7]:
from z3 import (
    ArithRef,
    Int,
    is_app,
    is_expr,
)

x: ArithRef = Int("x")
n: ArithRef = x + 1

print("is expression:", is_expr(x))
print("is application:", is_app(n))

print()

print("decl:    ", n.decl())
print("num args:", n.num_args())

for i in range(n.num_args()):
    print(f"arg({i}):", n.arg(i))

is expression: True
is application: True

decl:     +
num args: 2
arg(0): x
arg(1): 1


Declaration：関数記号そのものを調べる

In [8]:
from z3 import (
    ArithRef,
    BoolRef,
    BoolSort,
    FuncDeclRef,
    Function,
    Int,
    IntSort,
    RealSort,
    eq,
    is_expr,
    is_func_decl,
)

x: ArithRef = Int("x")

# 定数 x の Declaration
x_d: FuncDeclRef = x.decl()

print(is_expr(x_d))  # False
print(is_func_decl(x_d))  # True

print(x_d.name())  # x
print(x_d.range())  # Int
print(x_d.arity())  # 0
print(eq(x_d(), x))  # True

# (Int, Real) -> Bool
f: FuncDeclRef = Function("f", IntSort(), RealSort(), BoolSort())

print(f.name())  # f
print(f.range())  # Bool
print(f.arity())  # 2

for i in range(f.arity()):
    print(f"domain({i}):", f.domain(i))

app: BoolRef = f(x, x)
print(app)
print(eq(app.decl(), f))  # True

False
True
x
Int
0
True
f
Bool
2
domain(0): Int
domain(1): Real
f(x, ToReal(x))
True


In [9]:
x_d: FuncDeclRef = x.decl()

print(x_d.arity())  # 0
print(eq(x_d(), x))  # True

0
True


In [10]:
from z3 import (
    ArithRef,
    Int,
    Z3_OP_ADD,
    Z3_OP_SUB,
)

x: ArithRef = Int("x")
y: ArithRef = Int("y")

kind: int = (x + y).decl().kind()

print(kind == Z3_OP_ADD)  # True
print(kind == Z3_OP_SUB)  # False

True
False


substitutions

In [11]:
from z3 import (
    ArithRef,
    FuncDeclRef,
    Function,
    Int,
    IntSort,
    substitute,
)

x: ArithRef = Int("x")
y: ArithRef = Int("y")

f: FuncDeclRef = Function("f", IntSort(), IntSort(), IntSort())
g: FuncDeclRef = Function("g", IntSort(), IntSort())

n: ArithRef = f(
    f(g(x), g(g(x))),
    g(g(y)),
)

result: ArithRef = substitute(
    n,
    (g(g(x)), y),
    (g(y), x + 1),
)

print("Original:", n)
print("Result:  ", result)

Original: f(f(g(x), g(g(x))), g(g(y)))
Result:   f(f(g(x), y), g(x + 1))


declares a constant

In [12]:
Int("x")
Real("y")

y

In [13]:
from z3 import (
    And,
    BoolRef,
    BoolSort,
    Const,
    Int,
    IntSort,
    eq,
)

x = Const("x", IntSort())

print(eq(x, Int("x")))  # True

a: BoolRef = Const("a", BoolSort())
b: BoolRef = Const("b", BoolSort())

print(And(a, b))  # And(a, b)

True
And(a, b)


## Arrays

Z3 の Array は、巨大な配列や、インデックスの範囲が限定されていない配列をモデル化するためのもの

$$
\boxed{A:I\to V}
$$

- $I$：インデックスの Sort
- $V$：値の Sort

$$
A'=\operatorname{Store}(A,i,v)
$$

としたとき、
$$
\boxed{
A'[j]=
\begin{cases}
v & j=i\\
A[j] & j\ne i
\end{cases}
}
$$

In [14]:
from z3 import (
    ArithRef,
    Array,
    ArrayRef,
    Int,
    IntSort,
    Select,
    Store,
    SortRef,
    simplify,
)

I: SortRef = IntSort()

# 整数をインデックス、整数を値とする配列
A: ArrayRef = Array("A", I, I)

x: ArithRef = Int("x")

# 読み出し
print(A[x])
print(Select(A, x))

# 更新した新しい配列を構築
updated: ArrayRef = Store(A, x, 10)
print(updated)

# 書き込んだ位置を読み出す
result: ArithRef = simplify(Select(Store(A, 2, x + 1), 2))
print(result)

A[x]
A[x]
Store(A, x, 10)
1 + x


Array Extensionality（配列の外延性）
$$
A=B
\iff
\forall i,\ A[i]=B[i]
$$

---
EUF の関数との違い  
EUF の非解釈関数 $f,g$ について、
$$
\forall x,\ f(x)=g(x)
$$

ということだけで、関数記号 $f,g$ 自体を等しい対象として比較する仕組みにはなっていません。

Array と IntVector の比較

In [15]:
from z3 import (
    ArithRef,
    Array,
    ArrayRef,
    BoolRef,
    IntSort,
    IntVector,
    Sum,
)

# 方法1：Z3 Array（この用途では非推奨）
X_array: ArrayRef = Array("x_array", IntSort(), IntSort())

array_condition: BoolRef = X_array[0] + X_array[1] + X_array[2] >= 0

print(array_condition)

# 方法2：独立した3個の整数変数（推奨）
X_vector: list[ArithRef] = IntVector("x", 3)

vector_condition: BoolRef = Sum(X_vector) >= 0

print(vector_condition)


x_array[0] + x_array[1] + x_array[2] >= 0
x__0 + x__1 + x__2 >= 0


## Select and Store

$$
A'=\operatorname{Store}(A,i,v)
$$

$A'$ は、$A$ のインデックス $i$ の値だけを $v$ に変更した配列です。

基本公理は2つ

① 書き込んだ位置を読み出す
$$
\boxed{
\operatorname{Select}(\operatorname{Store}(A,i,v),i)=v
}
$$

② それ以外の位置を読み出す
$$
\boxed{
i\ne j
\implies
\operatorname{Select}(\operatorname{Store}(A,i,v),j)
=\operatorname{Select}(A,j)
}
$$

In [19]:
from z3 import (
    ArithRef,
    Array,
    ArrayRef,
    Int,
    IntSort,
    ModelRef,
    Solver,
    Store,
    sat,
)

A: ArrayRef = Array("A", IntSort(), IntSort())
x: ArithRef = Int("x")
y: ArithRef = Int("y")

solver: Solver = Solver()
solver.add(
    A[x] == x,
    Store(A, x, y) == A,
)

result = solver.check()
print(result)  # sat

if result == sat:
    model: ModelRef = solver.model()
    print("x =", model.eval(x))
    print("y =", model.eval(y))
    print("A =", model.eval(A))

sat
x = 3
y = 3
A = Store(K(Int, 2), 3, 3)


In [20]:
solver: Solver = Solver()
solver.add(
    A[x] == x,
    Store(A, x, y) == A,
    x != y,
)

print(solver.check())  # unsat

unsat


#### Constant arrays

すべてのインデックスに同じ値 $v$ が格納されている配列を構築

例えば、
$$
A=K(\mathrm{Int},1)
$$

なら、
$$
\boxed{\forall i\in\mathbb Z,\quad A[i]=1}
$$

In [21]:
from z3 import (
    ArithRef,
    ArrayRef,
    Int,
    IntSort,
    IntVal,
    K,
    Solver,
)

# すべての整数インデックスに 1 を持つ配列
AllOne: ArrayRef = K(IntSort(), IntVal(1))

a: ArithRef = Int("a")
i: ArithRef = Int("i")

# 例1：a == AllOne[i]
s1: Solver = Solver()
s1.add(a == AllOne[i])
print(s1.check())  # sat

# 例2：さらに a != 1 を追加
s2: Solver = Solver()
s2.add(
    a == AllOne[i],
    a != 1,
)
print(s2.check())  # unsat


sat
unsat


## Datatypes


|種類|役割|例|
|---|---|---|
|Constructor|値を構築する|`cons`, `nil`|
|Accessor|構成要素を取り出す|`car`, `cdr`|
|Recognizer|どの Constructor で構築されたか判定する|`is_cons`, `is_nil`|

### 整数 List を定義

In [22]:
from z3 import (
    Datatype,
    DatatypeSortRef,
    ExprRef,
    FuncDeclRef,
    IntSort,
    is_expr,
    is_func_decl,
    is_sort,
    simplify,
)

# Phase 1: Datatype の構造を宣言
list_builder: Datatype = Datatype("IntList")

# cons: (Int, IntList) -> IntList
list_builder.declare(
    "cons",
    ("car", IntSort()),
    ("cdr", list_builder),
)

# nil: IntList
list_builder.declare("nil")

# Phase 2: Datatype を完成
IntList: DatatypeSortRef = list_builder.create()

print(is_sort(IntList))  # True

# Constructor と Accessor を取得
cons: FuncDeclRef = IntList.cons
car: FuncDeclRef = IntList.car
cdr: FuncDeclRef = IntList.cdr
nil: ExprRef = IntList.nil

print(is_func_decl(cons))  # True
print(is_expr(nil))  # True

# [10, 20] に相当するリスト
l1: ExprRef = cons(10, cons(20, nil))

print(l1)
print(simplify(cdr(l1)))
print(simplify(car(l1)))
print(simplify(l1 == nil))


True
True
True
cons(10, cons(20, nil))
cons(20, nil)
10
False


### 任意の Sort に対応する List

In [23]:
from z3 import (
    Datatype,
    DatatypeSortRef,
    ExprRef,
    IntSort,
    RealSort,
    SortRef,
)


def declare_list(element_sort: SortRef) -> DatatypeSortRef:
    builder: Datatype = Datatype(f"List_of_{element_sort.name()}")

    builder.declare(
        "cons",
        ("car", element_sort),
        ("cdr", builder),
    )
    builder.declare("nil")

    return builder.create()


# 3種類の List 型を作成
IntList: DatatypeSortRef = declare_list(IntSort())
RealList: DatatypeSortRef = declare_list(RealSort())
IntListList: DatatypeSortRef = declare_list(IntList)

# 整数 List
l1: ExprRef = IntList.cons(10, IntList.nil)
print(l1)

# 整数 List を要素とする List
nested: ExprRef = IntListList.cons(
    l1,
    IntListList.cons(l1, IntListList.nil),
)
print(nested)

# 実数 List
real_list: ExprRef = RealList.cons(
    "1/3",
    RealList.nil,
)
print(real_list)

print(l1.sort())


cons(10, nil)
cons(cons(10, nil), cons(cons(10, nil), nil))
cons(1/3, nil)
List_of_Int


### Enumeration（列挙型）

In [24]:
from z3 import (
    BoolRef,
    Const,
    Datatype,
    DatatypeSortRef,
    ExprRef,
    Or,
    is_expr,
    prove,
    simplify,
)

builder: Datatype = Datatype("Color")

builder.declare("red")
builder.declare("green")
builder.declare("blue")

Color: DatatypeSortRef = builder.create()

print(is_expr(Color.green))  # True

# 等式を構築
equality: BoolRef = Color.green == Color.blue
print(equality)

# 等式を簡約
print(simplify(equality))  # False

# Color 型の任意の定数
c: ExprRef = Const("c", Color)

# c は必ず3色のいずれか
claim: BoolRef = Or(
    c == Color.red,
    c == Color.green,
    c == Color.blue,
)

prove(claim)  # proved

True
green == blue
False
proved


#### EnumSort() による省略記法

In [25]:
from z3 import (
    Const,
    DatatypeSortRef,
    EnumSort,
    ExprRef,
    Solver,
    simplify,
    sat,
)

Color: DatatypeSortRef
colors: list[ExprRef]

Color, colors = EnumSort(
    "ColorEnum",
    ("red", "green", "blue"),
)

red, green, blue = colors

print(green == blue)
print(simplify(green == blue))  # False

c: ExprRef = Const("c", Color)

solver: Solver = Solver()
solver.add(
    c != green,
    c != blue,
)

result = solver.check()
print(result)  # sat

if result == sat:
    print(solver.model().eval(c))  # red

green == blue
False
sat
red


### Mutually Recursive Datatypes（相互再帰型）

In [26]:
from z3 import (
    Consts,
    CreateDatatypes,
    Datatype,
    DatatypeSortRef,
    Distinct,
    ExprRef,
    IntSort,
    Solver,
    simplify,
)

# 2つの型の宣言を準備
tree_list_builder: Datatype = Datatype("TreeList")
tree_builder: Datatype = Datatype("Tree")

# Tree
tree_builder.declare(
    "leaf",
    ("val", IntSort()),
)
tree_builder.declare(
    "node",
    ("left", tree_list_builder),
    ("right", tree_list_builder),
)

# TreeList
tree_list_builder.declare("nil")
tree_list_builder.declare(
    "cons",
    ("car", tree_builder),
    ("cdr", tree_list_builder),
)

# 2つを同時に完成させる
Tree: DatatypeSortRef
TreeList: DatatypeSortRef

Tree, TreeList = CreateDatatypes(
    tree_builder,
    tree_list_builder,
)

# 値を構築
t1: ExprRef = Tree.leaf(10)
tl1: ExprRef = TreeList.cons(t1, TreeList.nil)
t2: ExprRef = Tree.node(tl1, TreeList.nil)

print(t2)
print(simplify(Tree.val(t1)))  # 10

# TreeList の異なる3つの値は存在できるか？
a: ExprRef
b: ExprRef
c: ExprRef
a, b, c = Consts("a b c", TreeList)

solver: Solver = Solver()
solver.add(Distinct(a, b, c))

print(solver.check())  # sat

node(cons(leaf(10), nil), nil)
10
sat


### Datatype と Sequence の組合せ

In [27]:
from z3 import (
    Datatype,
    DatatypeSort,
    DatatypeSortRef,
    SeqSort,
)

builder: Datatype = Datatype("SomeType")

# 完成前の Datatype Sort を参照
forward_sort: DatatypeSortRef = DatatypeSort("SomeType")

builder.declare("nil")
builder.declare(
    "some",
    ("someof", SeqSort(forward_sort)),
)

# Datatype の完成
SomeType: DatatypeSortRef = builder.create()

print(SomeType)


SomeType


### Uninterpreted Sorts

非解釈 Sort $A$、定数 $x,y$、非解釈関数 $f$ を宣言
$$
x,y:A,\qquad f:A\to A
$$

そのうえで、次の制約を課す
$$
\boxed{
\begin{aligned}
f(f(x))&=x\\
f(x)&=y\\
x&\ne y
\end{aligned}
}
$$

`model[A]`: モデル $M$ において、非解釈 Sort $A$ に割り当てられた議論領域を取得
$$
\boxed{D_A^M}
$$

In [28]:
from z3 import (
    Consts,
    DeclareSort,
    ExprRef,
    FuncDeclRef,
    Function,
    ModelRef,
    Solver,
    SortRef,
    sat,
)

# 非解釈 Sort
A: SortRef = DeclareSort("A")

# A に属する定数
x: ExprRef
y: ExprRef
x, y = Consts("x y", A)

# 非解釈関数 f: A -> A
f: FuncDeclRef = Function("f", A, A)

solver: Solver = Solver()
solver.add(
    f(f(x)) == x,
    f(x) == y,
    x != y,
)

result = solver.check()
print(result)  # sat

if result == sat:
    model: ModelRef = solver.model()

    print("Model:")
    print(model)

    print("Universe of A:")
    print(model[A])

    print("x =", model.eval(x))
    print("y =", model.eval(y))
    print("f(x) =", model.eval(f(x)))
    print("f(y) =", model.eval(f(y)))


sat
Model:
[x = A!val!0,
 y = A!val!1,
 f = [A!val!1 -> A!val!0, else -> A!val!1]]
Universe of A:
[A!val!0, A!val!1]
x = A!val!0
y = A!val!1
f(x) = A!val!1
f(y) = A!val!0


## Quantifiers

#### ForAll() の内部構造

$$
\boxed{
\forall x\forall y\;f(x,y)=0
}
$$

束縛変数は内部で Var(index, sort) に変換される
$$
\boxed{
\begin{aligned}
x&\longmapsto \mathrm{Var}(1)\\
y&\longmapsto \mathrm{Var}(0)
\end{aligned}
}
$$

In [29]:
from z3 import (
    ArithRef,
    ExprRef,
    ForAll,
    FuncDeclRef,
    Function,
    Int,
    IntSort,
    QuantifierRef,
    Var,
    eq,
)

# f: Int × Int -> Int
f: FuncDeclRef = Function("f", IntSort(), IntSort(), IntSort())

x: ArithRef = Int("x")
y: ArithRef = Int("y")

# ∀x∀y. f(x, y) = 0
axiom: QuantifierRef = ForAll(
    [x, y],
    f(x, y) == 0,
)

# 量化された式の本体を取得
body: ExprRef = axiom.body()
print("Body:", body)

# f の第1引数を取り出す
v1: ExprRef = body.arg(0).arg(0)

print("First argument:", v1)
print(eq(v1, Var(1, IntSort())))

Body: f(Var(1), Var(0)) == 0
First argument: Var(1)
True


実際どう使う？

$$
\forall x\;P(x)
$$

そして、
$$
\neg P(a)
$$

を追加

$$
\forall x\;P(x)\implies P(a)
$$

だからこの2つは矛盾

In [30]:
from z3 import (
    ArithRef,
    BoolSort,
    Exists,
    ForAll,
    FuncDeclRef,
    Function,
    Int,
    IntSort,
    Not,
    Solver,
)

P: FuncDeclRef = Function("P", IntSort(), BoolSort())

x: ArithRef = Int("x")
a: ArithRef = Int("a")

# 全称量化
s1: Solver = Solver()
s1.add(
    ForAll([x], P(x)),
    Not(P(a)),
)

print(s1.check())  # unsat

# 存在量化
s2: Solver = Solver()
s2.add(
    Exists([x], P(x)),
    Not(P(a)),
)

print(s2.check())  # sat


unsat
sat


### Modeling with Quantifiers


|公理|数式|意味|
|---|---|---|
|1. 反射律|$\forall x,\ x\preceq x$|すべての型は自分自身のサブタイプ|
|2. 推移律|$\forall x,y,z,\ (x\preceq y\land y\preceq z)\Rightarrow x\preceq z$|継承関係は推移する|
|3. 反対称律|$\forall x,y,\ (x\preceq y\land y\preceq x)\Rightarrow x=y$|相互にサブタイプなら同じ型|
|4. 単一継承|$\forall x,y,z,\ (x\preceq y\land x\preceq z)\Rightarrow(y\preceq z\lor z\preceq y)$|同じ型の上位型同士は比較可能|
|5. 単調性|$\forall x,y,\ x\preceq y\Rightarrow\operatorname{array\_of}(x)\preceq\operatorname{array\_of}(y)$|配列型の生成はサブタイプ関係を保存|
|6. root|$\forall x,\ root\preceq x$|root はあらゆる型の下位型|

In [33]:
from z3 import (
    And,
    BoolRef,
    BoolSort,
    Const,
    DeclareSort,
    ExprRef,
    ForAll,
    FuncDeclRef,
    Function,
    Implies,
    ModelRef,
    Or,
    Solver,
    SortRef,
    sat,
)

# 型を表す非解釈 Sort
Type: SortRef = DeclareSort("Type")

# サブタイプ関係: Type × Type → Bool
subtype: FuncDeclRef = Function("subtype", Type, Type, BoolSort())

# 配列型コンストラクタ: Type → Type
array_of: FuncDeclRef = Function("array_of", Type, Type)

# 基準となる型
root: ExprRef = Const("root", Type)

# 量化変数として使う定数
x: ExprRef = Const("x", Type)
y: ExprRef = Const("y", Type)
z: ExprRef = Const("z", Type)

axioms: list[BoolRef] = [
    # 1. 反射律
    ForAll(x, subtype(x, x)),
    # 2. 推移律
    ForAll(
        [x, y, z],
        Implies(
            And(subtype(x, y), subtype(y, z)),
            subtype(x, z),
        ),
    ),
    # 3. 反対称律
    ForAll(
        [x, y],
        Implies(
            And(subtype(x, y), subtype(y, x)),
            x == y,
        ),
    ),
    # 4. 単一継承
    ForAll(
        [x, y, z],
        Implies(
            And(subtype(x, y), subtype(x, z)),
            Or(subtype(y, z), subtype(z, y)),
        ),
    ),
    # 5. 配列型の単調性
    ForAll(
        [x, y],
        Implies(
            subtype(x, y),
            subtype(array_of(x), array_of(y)),
        ),
    ),
    # 6. root に関する制約
    # ForAll(x, subtype(root, x)),  # （原文のまま）
    ForAll(x, subtype(x, root)),  # 正しそう？
]

solver: Solver = Solver()
solver.add(axioms)

print(solver)
print(solver.check())

if solver.check() == sat:
    model: ModelRef = solver.model()
    print("Interpretation for Type:")
    print(model[Type])
    print()
    print("Model:")
    print(model)


[ForAll(x, subtype(x, x)),
 ForAll([x, y, z],
        Implies(And(subtype(x, y), subtype(y, z)),
                subtype(x, z))),
 ForAll([x, y],
        Implies(And(subtype(x, y), subtype(y, x)), x == y)),
 ForAll([x, y, z],
        Implies(And(subtype(x, y), subtype(x, z)),
                Or(subtype(y, z), subtype(z, y)))),
 ForAll([x, y],
        Implies(subtype(x, y),
                subtype(array_of(x), array_of(y)))),
 ForAll(x, subtype(x, root))]
sat
Interpretation for Type:
[Type!val!0]

Model:
[root = Type!val!0,
 subtype = [else -> True],
 array_of = [else -> Type!val!0]]


### Patterns

量化子のインスタンス化（Instantiation）: 全称量化された変数に具体的な項を代入する操作

In [36]:
from z3 import (
    ArithRef,
    BoolRef,
    FuncDeclRef,
    Function,
    ForAll,
    Int,
    IntSort,
    Solver,
)

# 非解釈関数 f, g: Int → Int
f: FuncDeclRef = Function("f", IntSort(), IntSort())
g: FuncDeclRef = Function("g", IntSort(), IntSort())

a: ArithRef = Int("a")
b: ArithRef = Int("b")
c: ArithRef = Int("c")
x: ArithRef = Int("x")

# 公理: ∀x. f(g(x)) = x
axiom: BoolRef = ForAll(
    x,
    f(g(x)) == x,
    # patterns : 論理式をいつ、どの項で具体化するかについてのヒント
    # patterns=[f(g(x))],  # 不適
    patterns=[g(x)],  # ok → 解ける
)

solver: Solver = Solver()

# Pattern の影響を観察するため、
# モデルベースのインスタンス化を無効化
solver.set(auto_config=False, mbqi=False)

solver.add(
    axiom,
    g(a) == c,
    g(b) == c,
    a != b,
)

print(solver.sexpr())
print()
print(solver.check())

(declare-fun g (Int) Int)
(declare-fun f (Int) Int)
(declare-fun c () Int)
(declare-fun a () Int)
(declare-fun b () Int)
(assert (forall ((x Int)) (! (= (f (g x)) x) :pattern ((g x)))))
(assert (= (g a) c))
(assert (= (g b) c))
(assert (distinct a b))


unsat


#### Matching Loop — インスタンス化が止まらない問題

$$
\forall x,y,\quad
subtype(x,y)\Rightarrow
subtype(array\_of(x),array\_of(y))
$$

Pattern に subtype(x, y) を指定したので、例えば探索中に
$$
subtype(Dog,Animal)
$$

という項が登場すると、Z3 は次のインスタンスを生成します。
$$
subtype(Dog,Animal)
\Rightarrow
subtype(array\_of(Dog),array\_of(Animal))
$$

しかし、生成された右辺にも subtype が含まれています。

In [ ]:
x: ExprRef = Const("x", Type)
y: ExprRef = Const("y", Type)

axiom = ForAll(
    [x, y],
    Implies(
        subtype(x, y),
        subtype(array_of(x), array_of(y)),
    ),
    patterns=[subtype(x, y)],
)


#### E-matching — 等式を考慮したパターンマッチング

$$
f(g(x))
\quad\overset{E}{\sim}\quad f(a)
$$

In [ ]:
from z3 import (
    ArithRef,
    BoolRef,
    FuncDeclRef,
    Function,
    ForAll,
    Int,
    IntSort,
    Solver,
)

f: FuncDeclRef = Function("f", IntSort(), IntSort())
g: FuncDeclRef = Function("g", IntSort(), IntSort())

a: ArithRef = Int("a")
b: ArithRef = Int("b")
c: ArithRef = Int("c")
x: ArithRef = Int("x")

axiom: BoolRef = ForAll(
    x,
    f(g(x)) == x,
    patterns=[f(g(x))],
)

solver: Solver = Solver()
solver.set(auto_config=False, mbqi=False)

solver.add(
    axiom,
    a == g(b),  # 
    b == c,
    f(a) != c,  # 
)

print(solver.check())
# unsat

unsat


### Multi-patterns

In [ ]:
from z3 import (
    BoolRef,
    Const,
    DeclareSort,
    ExprRef,
    ForAll,
    FuncDeclRef,
    Function,
    Implies,
    MultiPattern,
    Solver,
    SortRef,
)

# 非解釈 Sort
A: SortRef = DeclareSort("A")
B: SortRef = DeclareSort("B")

# f: A → B
f: FuncDeclRef = Function("f", A, B)

# 定数
a1: ExprRef = Const("a1", A)
a2: ExprRef = Const("a2", A)
b: ExprRef = Const("b", B)

# 量化変数
x: ExprRef = Const("x", A)
y: ExprRef = Const("y", A)

# f の単射性
injective: BoolRef = ForAll(
    [x, y],
    Implies(f(x) == f(y), x == y),
    patterns=[MultiPattern(f(x), f(y))],  # マルチパターン！
)

solver: Solver = Solver()
solver.add(
    a1 != a2,
    f(a1) == b,
    f(a2) == b,
    injective,
)

print(solver.check())
# unsat

unsat


左逆関数による表現

In [ ]:
from z3 import (
    BoolRef,
    Const,
    DeclareSort,
    ExprRef,
    ForAll,
    FuncDeclRef,
    Function,
    Solver,
    SortRef,
)

A: SortRef = DeclareSort("A")
B: SortRef = DeclareSort("B")

f: FuncDeclRef = Function("f", A, B)

# 左逆関数
finv: FuncDeclRef = Function("finv", B, A)

a1: ExprRef = Const("a1", A)
a2: ExprRef = Const("a2", A)
b: ExprRef = Const("b", B)
x: ExprRef = Const("x", A)

# 単射性を左逆関数で表現
injective: BoolRef = ForAll(
    x,
    finv(f(x)) == x,
    patterns=[f(x)],  # 1個で済むように
)

solver: Solver = Solver()
solver.add(
    a1 != a2,
    f(a1) == b,
    f(a2) == b,
    injective,
)

print(solver.check())
# unsat

unsat


### Other attributes

In [42]:
from z3 import (
    ArithRef,
    BoolRef,
    ForAll,
    FuncDeclRef,
    Function,
    Int,
    IntSort,
    Solver,
)

x: ArithRef = Int("x")

f: FuncDeclRef = Function("f", IntSort(), IntSort())
g: FuncDeclRef = Function("g", IntSort(), IntSort())

# qid, weight, no_patterns を設定
axiom: BoolRef = ForAll(
    x,
    g(f(x)) == x,
    qid="left_inverse_axiom",
    weight=5,
    no_patterns=[f(x)],
)

# SMT-LIB 形式で属性を確認
print(axiom.sexpr())

solver: Solver = Solver()
solver.set("smt.qi.profile", True)

a: ArithRef = Int("a")

solver.add(
    axiom,
    g(f(a)) != a,
)

print("Result:", solver.check())
print("Statistics:", solver.statistics())


(forall ((x Int))
  (! (= (g (f x)) x) :no-pattern (f x) :weight 5 :qid left_inverse_axiom))
Result: unsat
Statistics: (:arith-make-feasible  1
 :arith-max-columns    4
 :conflicts            1
 :max-generation       1
 :max-memory           20.43
 :memory               18.75
 :mk-bool-var          2
 :num-allocs           40977320
 :num-checks           2
 :quant-instantiations 1
 :rlimit-count         21422
 :time                 0.00)


[quantifier_instances] left_inverse_axiom :      1 :   0 :   0 :   0 : 1


## Multiple Solvers

In [43]:
from z3 import ArithRef, Int, Solver

x: ArithRef = Int("x")
y: ArithRef = Int("y")

# 1つ目の Solver
s1: Solver = Solver()
s1.add(x > 10, y > 10)

# 2つ目の Solver（初期状態は空）
s2: Solver = Solver()
print(s2)
# []

# s1 の制約を s2 にコピー
s2.add(s1.assertions())

print(s2)
# [x > 10, y > 10]

[]
[x > 10, y > 10]


## Unsat Cores and Soft Constraints

In [ ]:
from z3 import (
    ArithRef,
    Bool,
    BoolRef,
    Implies,
    Int,
    Solver,
    sat,
    unsat,
)

p1: BoolRef = Bool("p1")
p2: BoolRef = Bool("p2")
p3: BoolRef = Bool("p3")

x: ArithRef = Int("x")
y: ArithRef = Int("y")

solver: Solver = Solver()

solver.add(
    # ※ p1 は2つの制約をまとめて有効化する
    Implies(p1, x > 10),
    Implies(p1, y > x),
    Implies(p2, y < 5),
    Implies(p3, y > 0),
)

# すべてのラベルを有効化
result = solver.check(p1, p2, p3)
print(result)  # unsat

if result == unsat:
    print(solver.unsat_core())
    # {p1, p2}（表示順序は問わない）
    
print()

# p2 を有効化しない
result = solver.check(p1, p3)
print(result)  # sat

if result == sat:
    print(solver.model())

unsat
[p1, p2]

sat
[p2 = False, p1 = True, p3 = True, y = 12, x = 11]


### ついでに：Optimization

In [46]:
from z3 import (
    And,
    Bool,
    BoolRef,
    CheckSatResult,
    ModelRef,
    Not,
    Optimize,
    sat,
)

on_table: BoolRef = Bool("on_table")
on_shelf: BoolRef = Bool("on_shelf")

opt: Optimize = Optimize()

# Hard Constraint:
# 物体は同時に2か所に存在できない
opt.add(Not(And(on_table, on_shelf)))

# Soft Constraints:
# 観測Aを維持する優先度 = 3
opt.add_soft(on_table, weight=3)

# 観測Bを維持する優先度 = 1
opt.add_soft(on_shelf, weight=1)

result: CheckSatResult = opt.check()

if result == sat:
    model: ModelRef = opt.model()
    print(model.eval(on_table))  # True
    print(model.eval(on_shelf))  # False


True
False


## Formatter

In [47]:
from z3 import ArithRef, BoolRef, Int, set_option

x: ArithRef = Int("x")
y: ArithRef = Int("y")

formula: BoolRef = x**2 + y**2 >= 1

# HTML向けの数学的表記
set_option(html_mode=True)
print(formula)

# Z3Py形式の表記
set_option(html_mode=False)
print(formula)


x<sup>2</sup> + y<sup>2</sup> &ge; 1
x**2 + y**2 >= 1


In [48]:
from z3 import (
    ArithRef,
    BoolRef,
    And,
    IntVector,
    Sum,
    set_option,
)

x: list[ArithRef] = list(IntVector("x", 20))
y: list[ArithRef] = list(IntVector("y", 20))

formula: BoolRef = And(
    Sum(x) >= 0,
    Sum(y) >= 0,
)

# Test 1：各ノードの引数表示を制限
set_option(max_args=5)
print("\nTest 1:")
print(formula)

# Test 2：引数の上限を増やし、行数を制限
set_option(max_args=100, max_lines=10)
print("\nTest 2:")
print(formula)

# Test 3：1行の幅を広くする
set_option(max_width=300)
print("\nTest 3:")
print(formula)



Test 1:
And(x__0 + x__1 + x__2 + x__3 + x__4 + x__5 + ... >= 0,
    y__0 + y__1 + y__2 + y__3 + y__4 + y__5 + ... >= 0)

Test 2:
And(x__0 +
    x__1 +
    x__2 +
    x__3 +
    x__4 +
    x__5 +
    x__6 +
    x__7 +
    x__8 +
    x__9 +
...

Test 3:
And(x__0 + x__1 + x__2 + x__3 + x__4 + x__5 + x__6 + x__7 + x__8 + x__9 + x__10 + x__11 + x__12 + x__13 + x__14 + x__15 + x__16 + x__17 + x__18 + x__19 >= 0,
    y__0 + y__1 + y__2 + y__3 + y__4 + y__5 + y__6 + y__7 + y__8 + y__9 + y__10 + y__11 + y__12 + y__13 + y__14 + y__15 + y__16 + y__17 + y__18 + y__19 >= 0)


In [49]:
from z3 import ArithRef, RealVal, set_option

value: ArithRef = RealVal("1/3")

# 分数として表示
set_option(rational_to_decimal=False)
print(value)

# 小数として表示
set_option(
    rational_to_decimal=True,
    precision=10,
)
print(value)


1/3
0.3333333333?
